In [0]:
from pyspark.sql.functions import *

In [0]:
import sys
import importlib
sys.path.append('/Workspace/Users/kiranmanam9490@outlook.com/customer360')

import common.configuration
importlib.reload(common.configuration)
common.configuration.dbutils = dbutils

from common.configuration import Configuration

conf = Configuration()

In [0]:
df = spark.read.format('parquet').load(conf.base_url + '/bronze/customers')


In [0]:
df = df.withColumn(
    "dob",
    coalesce(
        try_to_date(col("dob"), "yyyy/MM/dd"),
        try_to_date(col("dob"), "dd-MM-yyyy"),
        try_to_date(col("dob"), "yyyy-MM-dd")
    )
)

In [0]:
df = df.withColumnRenamed("EMAIL", "email")\
    .withColumn('email',lower(col("email")))\
        .withColumn("name",initcap(col("name")))\
            .withColumn("location",initcap(col("location")))

In [0]:
df = df.withColumn("gender",initcap(when(col('gender').isin(["F","f"]) ,"Female")\
    .when(col('gender').isin(["M","m"]) ,"Male")\
        .when(col('gender').isNull(),"Unknown")\
        .otherwise(col('gender'))))

In [0]:
df = df.dropDuplicates(["customer_id"])

In [0]:
df =df.dropna(subset=["customer_id","email","name"])

In [0]:

df = df.withColumn("customer_id", col("customer_id").cast("int")).withColumn("dob", col("dob").cast("date"))

In [0]:
df.write.format("delta").mode("overwrite").save(conf.base_url + '/silver/customers')